# Intelligent Loan Approval Prediction using Random Forest

## User Story
As a bank manager, I want a system that analyzes customer details like age, income, credit score, loan amount, and loan term, so that I can quickly and accurately decide whether to approve or reject a loan, reducing risk and manual effort.

## Scenario
A customer applies for a loan. Instead of manually reviewing every application, this notebook uses a Random Forest model trained on historical data to predict whether the application should be approved.

## 1. Import Libraries and Load Historical Loan Data

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

import joblib

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

DATA_PATH = "loan_data.csv"

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded dataset from {DATA_PATH} with shape: {df.shape}")
else:
    rng = np.random.default_rng(42)
    n_samples = 2000

    age = rng.integers(21, 66, n_samples)
    income = rng.normal(65000, 18000, n_samples).clip(18000, 180000)
    credit_score = rng.normal(680, 85, n_samples).clip(300, 850)
    loan_amount = rng.normal(180000, 70000, n_samples).clip(5000, 500000)
    loan_term = rng.choice([12, 24, 36, 48, 60, 72, 84], n_samples, p=[0.05, 0.1, 0.3, 0.2, 0.2, 0.1, 0.05])
    employment_years = rng.integers(0, 31, n_samples)
    existing_debt = rng.normal(20000, 15000, n_samples).clip(0, 150000)
    employment_type = rng.choice(["salaried", "self_employed", "contract", "unemployed"], n_samples, p=[0.55, 0.2, 0.2, 0.05])

    # Synthetic risk score to generate approval labels.
    risk_score = (
        0.005 * (loan_amount / 1000)
        + 0.015 * (existing_debt / 1000)
        + 0.02 * np.maximum(650 - credit_score, 0)
        + 0.08 * (loan_term / 12)
        - 0.03 * (income / 10000)
        - 0.04 * employment_years
        + np.where(employment_type == "unemployed", 1.8, 0)
        + np.where(employment_type == "contract", 0.4, 0)
    )

    approval_probability = 1 / (1 + np.exp(risk_score - 2.8))
    loan_approved = (approval_probability > 0.5).astype(int)

    df = pd.DataFrame(
        {
            "age": age,
            "income": income.round(2),
            "credit_score": credit_score.round(0).astype(int),
            "loan_amount": loan_amount.round(2),
            "loan_term": loan_term,
            "employment_years": employment_years,
            "existing_debt": existing_debt.round(2),
            "employment_type": employment_type,
            "loan_approved": loan_approved,
        }
    )
    print(f"Generated synthetic dataset with shape: {df.shape}")

df.head()

## 2. Inspect Schema and Clean Missing/Invalid Values

In [ ]:
print("DataFrame shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

# Remove duplicates.
df = df.drop_duplicates().copy()

# Basic validity constraints for key numeric columns.
valid_ranges = {
    "age": (18, 100),
    "income": (0, 1_000_000),
    "credit_score": (300, 850),
    "loan_amount": (1000, 5_000_000),
    "loan_term": (6, 480),
    "employment_years": (0, 60),
    "existing_debt": (0, 5_000_000),
}

for col, (low, high) in valid_ranges.items():
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].clip(lower=low, upper=high)

# Impute missing values: median for numeric, mode for categorical.
for col in df.columns:
    if df[col].dtype in ["object", "category"]:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].mode(dropna=True).iloc[0])
    else:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())

# Optional outlier capping using IQR rule for major financial fields.
for col in ["income", "loan_amount", "existing_debt"]:
    if col in df.columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        df[col] = df[col].clip(lower=lower, upper=upper)

print("\nAfter cleaning:")
print("Missing values:\n", df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
df.head()

## 3. Encode Categorical Features and Define Target

In [ ]:
TARGET_COL = "loan_approved"

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in dataset.")

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].astype(int)

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_features = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)
print("Target distribution:\n", y.value_counts(normalize=True).rename("proportion"))

## 4. Split Data into Train and Test Sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target distribution:\n", y_train.value_counts(normalize=True).rename("proportion"))

## 5. Train Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", rf_model),
    ]
)

model_pipeline.fit(X_train, y_train)
print("Baseline Random Forest model trained successfully.")

## 6. Evaluate Model Performance

In [ ]:
y_pred = model_pipeline.predict(X_test)
y_proba = model_pipeline.predict_proba(X_test)[:, 1]

metrics_summary = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1_score": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba),
}

print("Baseline model metrics:")
for metric, value in metrics_summary.items():
    print(f"- {metric}: {value:.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, digits=4))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Confusion Matrix - Baseline Model")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

## 7. Predict Approval for New Customer Applications

In [ ]:
def predict_loan_approval(customer_dict, trained_pipeline=model_pipeline):
    customer_df = pd.DataFrame([customer_dict])
    pred_class = int(trained_pipeline.predict(customer_df)[0])
    pred_prob = float(trained_pipeline.predict_proba(customer_df)[0, 1])

    return {
        "prediction": "Approved" if pred_class == 1 else "Rejected",
        "approval_probability": pred_prob,
    }


sample_customer = {
    "age": 35,
    "income": 78000,
    "credit_score": 720,
    "loan_amount": 150000,
    "loan_term": 60,
    "employment_years": 8,
    "existing_debt": 12000,
    "employment_type": "salaried",
}

sample_prediction = predict_loan_approval(sample_customer)
print("Sample customer prediction:", sample_prediction)

## 8. Tune Hyperparameters with Cross-Validation

In [ ]:
param_grid = {
    "classifier__n_estimators": [200, 400],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=model_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

print("Best parameters:", grid_search.best_params_)
print(f"Best CV F1-score: {grid_search.best_score_:.4f}")

y_pred_best = best_model.predict(X_test)
y_proba_best = best_model.predict_proba(X_test)[:, 1]

print("\nTuned model test metrics:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_best):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_best, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_best, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred_best, zero_division=0):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba_best):.4f}")

## 9. Interpret Feature Importance for Risk Insights

In [ ]:
best_preprocessor = best_model.named_steps["preprocessor"]
best_classifier = best_model.named_steps["classifier"]

numeric_names = numerical_features.copy()

if len(categorical_features) > 0:
    onehot = best_preprocessor.named_transformers_["cat"].named_steps["onehot"]
    categorical_names = onehot.get_feature_names_out(categorical_features).tolist()
else:
    categorical_names = []

feature_names = numeric_names + categorical_names
importances = best_classifier.feature_importances_

feature_importance_df = pd.DataFrame(
    {
        "feature": feature_names,
        "importance": importances,
    }
).sort_values("importance", ascending=False)

print("Top 10 important features:")
display(feature_importance_df.head(10))

plt.figure(figsize=(10, 6))
sns.barplot(
    data=feature_importance_df.head(15),
    x="importance",
    y="feature",
    palette="viridis",
)
plt.title("Top Feature Importances - Tuned Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 10. Save and Reload the Trained Model

In [ ]:
MODEL_PATH = "loan_approval_rf_pipeline.joblib"

joblib.dump(best_model, MODEL_PATH)
print(f"Saved tuned model pipeline to: {MODEL_PATH}")

reloaded_model = joblib.load(MODEL_PATH)

sample_df = pd.DataFrame([sample_customer])
original_prob = float(best_model.predict_proba(sample_df)[0, 1])
reloaded_prob = float(reloaded_model.predict_proba(sample_df)[0, 1])

print(f"Original model probability: {original_prob:.6f}")
print(f"Reloaded model probability: {reloaded_prob:.6f}")
print("Inference consistency check:", np.isclose(original_prob, reloaded_prob))